# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [5]:
pip install tensorflow

  Using cached absl_py-2.5.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------------- 0.0/382.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/382.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/382.7 MB 3.6 MB/s eta 0:01:47
   ---------------------------------------- 0.5/382.7 MB 3.6 MB/s eta 0:01:47
   -----


[notice] A new release of pip is available: 26.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [7]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [8]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [9]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [11]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

device = "/cpu:0"       
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [12]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding='same',
            activation='relu'
        )
        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding='same',
            activation='relu'
        )
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(num_classes)
        self.softmax = tf.keras.layers.Softmax()

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)       # (N, H, W, channel_1)  ReLU applied inside
        x = self.conv2(x)       # (N, H, W, channel_2)  ReLU applied inside
        x = self.flatten(x)     # (N, H*W*channel_2)
        x = self.fc(x)          # (N, num_classes)
        scores = self.softmax(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [13]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [4]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False, print_every=50):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during trainingn
    """    
    with tf.device(device):

        
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    
                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    
                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [29]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.9961953163146973, Accuracy: 9.375, Val Loss: 3.043699264526367, Val Accuracy: 10.5
Iteration 50, Epoch 1, Loss: 2.4088234901428223, Accuracy: 24.32598114013672, Val Loss: 2.1767539978027344, Val Accuracy: 31.799999237060547
Iteration 100, Epoch 1, Loss: 2.258552074432373, Accuracy: 28.094058990478516, Val Loss: 1.851699709892273, Val Accuracy: 38.900001525878906
Iteration 150, Epoch 1, Loss: 2.1457622051239014, Accuracy: 30.691225051879883, Val Loss: 1.8580448627471924, Val Accuracy: 38.0
Iteration 200, Epoch 1, Loss: 2.07751727104187, Accuracy: 32.37717819213867, Val Loss: 1.821142554283142, Val Accuracy: 37.5
Iteration 250, Epoch 1, Loss: 2.0406267642974854, Accuracy: 33.22957992553711, Val Loss: 1.812269687652588, Val Accuracy: 38.5
Iteration 300, Epoch 1, Loss: 2.0029144287109375, Accuracy: 34.105064392089844, Val Loss: 1.9044166803359985, Val Accuracy: 36.39999771118164
Iteration 350, Epoch 1, Loss: 1.969781517982483, Accuracy: 34.962608337402344, Val

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [27]:
learning_rate = 7e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (no need to edit other parts of the code)*****
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    # *****END OF YOUR CODE*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (no need to edit other parts of the code)*****
    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )
    # *****END OF YOUR CODE*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.3712077140808105, Accuracy: 12.5, Val Loss: 2.320326328277588, Val Accuracy: 12.0
Iteration 50, Epoch 1, Loss: 1.9914783239364624, Accuracy: 29.105392456054688, Val Loss: 1.7122918367385864, Val Accuracy: 40.0
Iteration 100, Epoch 1, Loss: 1.8058841228485107, Accuracy: 35.78279495239258, Val Loss: 1.5453346967697144, Val Accuracy: 46.39999771118164
Iteration 150, Epoch 1, Loss: 1.7152035236358643, Accuracy: 39.18667221069336, Val Loss: 1.5072327852249146, Val Accuracy: 48.599998474121094
Iteration 200, Epoch 1, Loss: 1.660489559173584, Accuracy: 40.967041015625, Val Loss: 1.4307414293289185, Val Accuracy: 50.0
Iteration 250, Epoch 1, Loss: 1.621747374534607, Accuracy: 42.28087615966797, Val Loss: 1.3788090944290161, Val Accuracy: 50.80000305175781
Iteration 300, Epoch 1, Loss: 1.5830225944519043, Accuracy: 43.698089599609375, Val Loss: 1.3770549297332764, Val Accuracy: 51.0
Iteration 350, Epoch 1, Loss: 1.5512808561325073, Accuracy: 44.796119689941406, Val

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [28]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

c:\Users\vliva\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.2039952278137207, Accuracy: 10.9375, Val Loss: 2.940009832382202, Val Accuracy: 13.199999809265137
Iteration 50, Epoch 1, Loss: 2.383864402770996, Accuracy: 24.785539627075195, Val Loss: 2.217759847640991, Val Accuracy: 31.400001525878906
Iteration 100, Epoch 1, Loss: 2.2361910343170166, Accuracy: 27.815595626831055, Val Loss: 1.8902888298034668, Val Accuracy: 37.70000076293945
Iteration 150, Epoch 1, Loss: 2.144777297973633, Accuracy: 29.925495147705078, Val Loss: 1.9507054090499878, Val Accuracy: 36.599998474121094
Iteration 200, Epoch 1, Loss: 2.0824837684631348, Accuracy: 31.731966018676758, Val Loss: 1.8615063428878784, Val Accuracy: 39.5
Iteration 250, Epoch 1, Loss: 2.0451958179473877, Accuracy: 32.6630973815918, Val Loss: 1.8201993703842163, Val Accuracy: 38.20000076293945
Iteration 300, Epoch 1, Loss: 2.002532482147217, Accuracy: 33.892234802246094, Val Loss: 1.8819804191589355, Val Accuracy: 36.5
Iteration 350, Epoch 1, Loss: 1.9706920385360718, 

Альтернативный менее гибкий способ обучения:

In [30]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 1.8145 - sparse_categorical_accuracy: 0.3890 - val_loss: 1.6332 - val_sparse_categorical_accuracy: 0.4390
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 1.6195 - sparse_categorical_accuracy: 0.4432


[1.6195076704025269, 0.4431999921798706]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [32]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    input_shape = (32, 32, 3)
    channel_1, channel_2, num_classes = 32, 16, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0, seed=None)

    layers = [
        tf.keras.layers.Conv2D(channel_1, (5, 5), padding='same',
                               activation='relu', input_shape=input_shape,
                               kernel_initializer=initializer),
        tf.keras.layers.Conv2D(channel_2, (3, 3), padding='same',
                               activation='relu',
                               kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model


learning_rate = 5e-4

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate,
                                        momentum=0.9,
                                        nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

c:\Users\vliva\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 3.697006940841675, Accuracy: 7.8125, Val Loss: 2.9677674770355225, Val Accuracy: 11.699999809265137
Iteration 50, Epoch 1, Loss: 2.192089557647705, Accuracy: 25.12255096435547, Val Loss: 1.8937493562698364, Val Accuracy: 34.099998474121094
Iteration 100, Epoch 1, Loss: 2.008906602859497, Accuracy: 30.213489532470703, Val Loss: 1.773519515991211, Val Accuracy: 36.39999771118164
Iteration 150, Epoch 1, Loss: 1.924963355064392, Accuracy: 32.93667221069336, Val Loss: 1.7142024040222168, Val Accuracy: 41.0
Iteration 200, Epoch 1, Loss: 1.8665419816970825, Accuracy: 35.027984619140625, Val Loss: 1.6455134153366089, Val Accuracy: 42.79999923706055
Iteration 250, Epoch 1, Loss: 1.82370924949646, Accuracy: 36.311004638671875, Val Loss: 1.6088581085205078, Val Accuracy: 42.599998474121094
Iteration 300, Epoch 1, Loss: 1.7854105234146118, Accuracy: 37.45328140258789, Val Loss: 1.5962270498275757, Val Accuracy: 44.29999923706055
Iteration 350, Epoch 1, Loss: 1.753505468

In [33]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 1.6060 - sparse_categorical_accuracy: 0.4353 - val_loss: 1.4239 - val_sparse_categorical_accuracy: 0.4850
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.4312 - sparse_categorical_accuracy: 0.4923


[1.4312455654144287, 0.49230000376701355]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [10]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [35]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.0781702995300293, Accuracy: 3.125, Val Loss: 2.9149692058563232, Val Accuracy: 16.0
Iteration 50, Epoch 1, Loss: 2.357583522796631, Accuracy: 25.582107543945312, Val Loss: 2.1157281398773193, Val Accuracy: 31.299999237060547
Iteration 100, Epoch 1, Loss: 2.2293295860290527, Accuracy: 28.60457992553711, Val Loss: 1.9085439443588257, Val Accuracy: 38.400001525878906
Iteration 150, Epoch 1, Loss: 2.1310231685638428, Accuracy: 30.732616424560547, Val Loss: 1.904909372329712, Val Accuracy: 38.29999923706055
Iteration 200, Epoch 1, Loss: 2.073477268218994, Accuracy: 32.05845642089844, Val Loss: 1.834444522857666, Val Accuracy: 38.60000228881836
Iteration 250, Epoch 1, Loss: 2.0355710983276367, Accuracy: 33.080177307128906, Val Loss: 1.8160897493362427, Val Accuracy: 39.29999923706055
Iteration 300, Epoch 1, Loss: 1.997166633605957, Accuracy: 34.08949279785156, Val Loss: 1.859962821006775, Val Accuracy: 37.20000076293945
Iteration 350, Epoch 1, Loss: 1.9628899097

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same',
                                            activation='relu')
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same',
                                            activation='relu')
        self.pool1 = tf.keras.layers.MaxPool2D(pool_size=2)
        self.conv3 = tf.keras.layers.Conv2D(128, 3, padding='same',
                                            activation='relu')
        self.pool2 = tf.keras.layers.MaxPool2D(pool_size=2)
        self.conv4 = tf.keras.layers.Conv2D(256, 3, padding='same',
                                            activation='relu')
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.conv2(x)
        x = self.pool1(x)
        x = self.conv3(x)
        x = self.pool2(x)
        x = self.conv4(x)
        x = self.flatten(x)
        x = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x

print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 
device = "/cpu:0"
train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True, print_every=print_every)

Iteration 0, Epoch 1, Loss: 2.293104410171509, Accuracy: 15.625, Val Loss: 2.6515417098999023, Val Accuracy: 7.90000057220459
Iteration 700, Epoch 1, Loss: 1.2267043590545654, Accuracy: 56.27006149291992, Val Loss: 0.9028751850128174, Val Accuracy: 69.30000305175781
Iteration 1400, Epoch 2, Loss: 0.751305103302002, Accuracy: 73.8090591430664, Val Loss: 0.6854650378227234, Val Accuracy: 75.0999984741211
Iteration 2100, Epoch 3, Loss: 0.5625863075256348, Accuracy: 80.6129150390625, Val Loss: 0.7862475514411926, Val Accuracy: 74.4000015258789
Iteration 2800, Epoch 4, Loss: 0.4234071969985962, Accuracy: 85.36592864990234, Val Loss: 0.7229193449020386, Val Accuracy: 77.9000015258789
Iteration 3500, Epoch 5, Loss: 0.3169751465320587, Accuracy: 88.92305755615234, Val Loss: 0.8594271540641785, Val Accuracy: 77.9000015258789
Iteration 4200, Epoch 6, Loss: 0.2589069902896881, Accuracy: 90.742919921875, Val Loss: 0.8651896119117737, Val Accuracy: 76.9000015258789
Iteration 4900, Epoch 7, Loss: 0.

Опишите все эксперименты, результаты. Сделайте выводы.

In [ ]:
1) первый эксперимент 62%
1. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой 
6. Функция активации Softmax
2) второй эксперимент 68%
1. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. пул pool_size=2
6. Полносвязный слой 
7. Функция активации Softmax
3) третий эксперимент 71%
1. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. пул pool_size=2
6. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
7. Функция активации ReLU
8. пул pool_size=2
9. Полносвязный слой 
10. Функция активации Softmax
4) четвертый эксперимент 79%
1. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. пул pool_size=2
6. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
7. Функция активации ReLU
8. пул pool_size=2
9. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
10. Функция активации ReLU
11. Полносвязный слой 
12. Функция активации Softmax

последовательное увеличение глубины сети за счёт добавления свёрточных слоёв с пулингом дает рост точности